In [ ]:
from notebook.services.config import ConfigManagercm = ConfigManager()cm.update('livereveal', {    'width': 1920,    'height': 1080,    'scroll': True,})

# Week 11: Monday, AST 5011: Astrophysical Systems

## Stars & Stellar Populations

### Michael Coughlin

**Reference:** Cimatti, Fraternali & Nipoti, Ch. 7

With material from Benedikt Diemer (UMD) and Frank van den Bosch (Yale).

In [ ]:
import numpy as npimport scipy.integrateimport matplotlib.pyplot as pltfrom matplotlib.ticker import LogLocatorfrom colossus.cosmology import cosmologyimport routines as rt%matplotlib inline%config InlineBackend.figure_format = 'retina'cosmo = cosmology.setCosmology('planck18')

## The Initial Mass Function (IMF)The **initial mass function (IMF)** describes the distribution of stellar masses at birth. It is one of the most fundamental quantities in astrophysics, as it determines:- The ratio of massive to low-mass stars- The total luminosity per unit mass formed- The rate of supernova feedback- The chemical enrichment of the ISMThe IMF is typically expressed as $dn/d\log_{10} m$, the number of stars per logarithmic mass interval. Several fitting functions are commonly used:- **Salpeter (1955)**: $\xi(m) \propto m^{-1.35}$ (single power law)- **Kroupa (2001)**: broken power law with shallower slope below $0.5\,M_\odot$- **Chabrier (2003)**: lognormal below $1\,M_\odot$, power law aboveThe key difference: Salpeter predicts more low-mass stars relative to high-mass stars compared to Kroupa/Chabrier. This affects mass-to-light ratios by a factor of $\sim 1.5$–$2$.

In [ ]:
# Plot the three IMFsmmin, mmax = 0.08, 150.0m = 10**np.linspace(np.log10(mmin), np.log10(mmax), 200)imf_funcs = [rt.imfSalpeter, rt.imfKroupa, rt.imfChabrier]imf_labels = ['Salpeter 1955', 'Kroupa 2001', 'Chabrier 2003']# Normalize each IMF so that integral of m * xi(m) d(log m) = 1imfs_normed = []for func in imf_funcs:    xi = func(m)    # Normalize by total mass    integrand_func = lambda logm: func(10**logm) * 10**logm    norm, _ = scipy.integrate.quad(integrand_func, np.log10(mmin), np.log10(mmax))    imfs_normed.append(xi / norm)fig, ax = plt.subplots(figsize=(5, 4))ax.set_xscale('log')ax.set_yscale('log')ax.set_xlabel(r'$m\ (M_\odot)$')ax.set_ylabel(r'$\xi(m) = dn/d\log_{10} m$ (mass-normalized)')ax.set_xlim(mmin, mmax)for i, xi in enumerate(imfs_normed):    ax.plot(m, xi, label=imf_labels[i])ax.legend(fontsize=10)ax.set_title('Initial Mass Functions')plt.tight_layout()plt.show()

## Demonstration: Cumulative IMF Distributions

The differential IMF $\xi(m)$ tells us the number of stars per logarithmic mass interval, but the **cumulative** distributions reveal where the total number and mass reside:

- **$N(>m)/N_{\rm total}$**: fraction of stars more massive than $m$. Most stars by number are low-mass ($\sim 0.1$–$0.5\,M_\odot$).
- **$M(>m)/M_{\rm total}$**: fraction of total stellar mass in stars more massive than $m$. A significant fraction of the mass is in intermediate-mass stars ($\sim 1$–$8\,M_\odot$), even though they are outnumbered by low-mass stars.

The Salpeter IMF predicts more low-mass stars (steeper drop in cumulative number) compared to Kroupa/Chabrier, which have flatter low-mass slopes (adapted from CFN Chapter 7).

In [ ]:
# Cumulative IMF distributions: number and mass fractions
m_cum = 10**np.linspace(np.log10(0.08), np.log10(150.0), 300)
dlogm = np.log10(m_cum[1]) - np.log10(m_cum[0])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for j, (func, label) in enumerate(zip(imf_funcs, imf_labels)):
    xi = func(m_cum)
    # Cumulative from high mass: N(>m) = integral from m to m_max of xi dlogm
    N_cum = np.cumsum(xi[::-1])[::-1] * dlogm
    M_cum = np.cumsum((m_cum * xi)[::-1])[::-1] * dlogm

    axes[0].semilogx(m_cum, N_cum / N_cum[0], lw=2, label=label)
    axes[1].semilogx(m_cum, M_cum / M_cum[0], lw=2, label=label)

for ax in axes:
    ax.set_xlabel(r'$m\ (M_\odot)$')
    ax.set_xlim(0.08, 150)
    ax.set_ylim(0, 1.05)
    ax.axvline(8.0, ls=':', color='gray', lw=0.8)
    ax.text(9.0, 0.85, r'$8\,M_\odot$', color='gray', fontsize=9)
    ax.legend(fontsize=9)

axes[0].set_ylabel(r'$N(>m) / N_{\rm total}$')
axes[0].set_title('Cumulative Number Fraction')
axes[1].set_ylabel(r'$M(>m) / M_{\rm total}$')
axes[1].set_title('Cumulative Mass Fraction')

plt.tight_layout()
plt.show()

## Exercise 1: IMF Mass BudgetFor each IMF, compute:1. The **number fraction** of stars with $m > 8\,M_\odot$ (massive stars that explode as core-collapse supernovae)2. The **mass fraction** in massive stars ($m > 8\,M_\odot$)3. The **mean stellar mass** $\langle m \rangle$Use the mass range $0.08$–$150\,M_\odot$. The integrals are:$$f_{\text{number}} = \frac{\int_8^{150} \xi(m)\,d\log m}{\int_{0.08}^{150} \xi(m)\,d\log m}, \quad f_{\text{mass}} = \frac{\int_8^{150} m\,\xi(m)\,d\log m}{\int_{0.08}^{150} m\,\xi(m)\,d\log m}$$

In [ ]:
# Exercise 1: IMF mass budgetlog_mmin = np.log10(0.08)log_mmax = np.log10(150.0)log_m_massive = np.log10(8.0)for i, func in enumerate(imf_funcs):    # Number integrals: integrate xi(m) d(log m)    n_integrand = lambda logm: func(10**logm)    # FILL IN: integrate number of stars over full range and massive range    N_total, _ = ...  # FILL IN: scipy.integrate.quad(n_integrand, log_mmin, log_mmax)    N_massive, _ = ...  # FILL IN: scipy.integrate.quad(n_integrand, log_m_massive, log_mmax)    # Mass integrals: integrate m * xi(m) d(log m)    m_integrand = lambda logm: 10**logm * func(10**logm)    # FILL IN: integrate mass over full range and massive range    M_total, _ = ...  # FILL IN: scipy.integrate.quad(m_integrand, log_mmin, log_mmax)    M_massive, _ = ...  # FILL IN: scipy.integrate.quad(m_integrand, log_m_massive, log_mmax)    # FILL IN: compute mean stellar mass    mean_mass = ...  # FILL IN: M_total / N_total    f_num = N_massive / N_total    f_mass = M_massive / M_total    print(f'{imf_labels[i]:20s}: f_number(>8) = {f_num:.4f}, '          f'f_mass(>8) = {f_mass:.3f}, <m> = {mean_mass:.3f} Msun')

## Star Formation HistoriesThe star formation history (SFH) describes how the star formation rate of a galaxy evolves with time. Common parameterizations include:1. **Exponential decay**: $\psi(t) \propto e^{-t/\tau}$ — rapid initial burst followed by decline2. **Delayed-$\tau$ model**: $\psi(t) \propto (t/\tau)\,e^{-t/\tau}$ — rises to a peak then declines3. **Lognormal**: peaked SFH with asymmetric tailsThe parameter $\tau$ controls the timescale: small $\tau$ means a short burst (forming an "old, red" population), while large $\tau$ means extended star formation (a "young, blue" population).

In [ ]:
# Star formation history modelst_age = cosmo.age(0.0)  # age of Universe in Gyrt_plot = np.linspace(0.1, t_age, 200)fig, ax = plt.subplots(figsize=(5, 4))ax.set_xlabel(r'$t\ (\rm Gyr)$')ax.set_ylabel(r'SFR (arbitrary units)')ax.set_xlim(0, t_age)# Exponential decay with different taufor tau in [0.5, 2.0, 5.0]:    sfr = rt.sfrExponential(t_plot, tau)    sfr /= np.max(sfr)    ax.plot(t_plot, sfr, label=r'Exp, $\tau = %.1f$ Gyr' % tau)# Delayed-tausfr_del = rt.sfrDelayedTau(t_plot, 2.0)sfr_del /= np.max(sfr_del)ax.plot(t_plot, sfr_del, '--', label=r'Delayed-$\tau$, $\tau = 2$ Gyr')ax.legend(fontsize=9)ax.set_title('Star Formation Histories')plt.tight_layout()plt.show()

## Demonstration: Stellar Mass Assembly

The SFH determines **when** a galaxy assembles its stellar mass. The cumulative mass fraction $f_*(t) = \int_0^t \psi(t')\,dt' / \int_0^{t_{\rm age}} \psi(t')\,dt'$ shows how quickly mass builds up.

- **Short $\tau$** (burst): mass is assembled early, producing an old, red population
- **Long $\tau$** (extended SF): mass builds up gradually, resulting in a younger, bluer population
- **Delayed-$\tau$**: mass assembly peaks later, with a more gradual onset

This also connects to "downsizing" — more massive galaxies tend to have shorter effective $\tau$, forming their stars earlier despite residing in more massive halos (adapted from CFN Chapter 7).

In [ ]:
# Stellar mass assembly: cumulative mass fraction for different SFH models
t_age_gyr = cosmo.age(0.0)  # Gyr
t_assembly = np.linspace(0.01, t_age_gyr, 500)

fig, ax = plt.subplots(figsize=(5, 4))
ax.set_xlabel(r'$t\ (\rm Gyr)$')
ax.set_ylabel(r'$M_*(< t) / M_{*,\rm total}$')
ax.set_xlim(0, t_age_gyr)
ax.set_ylim(0, 1.05)

sfh_models = [
    (lambda t: rt.sfrExponential(t, 0.5), r'Exp, $\tau = 0.5$ Gyr', '-'),
    (lambda t: rt.sfrExponential(t, 2.0), r'Exp, $\tau = 2.0$ Gyr', '-'),
    (lambda t: rt.sfrExponential(t, 5.0), r'Exp, $\tau = 5.0$ Gyr', '-'),
    (lambda t: rt.sfrDelayedTau(t, 2.0), r'Delayed-$\tau$, $\tau = 2$ Gyr', '--'),
    (lambda t: rt.sfrLognormal(t, 3.0, 2.0), r'Lognormal ($t_0=3, \tau=2$)', ':'),
]

for sfh_func, label, ls in sfh_models:
    sfr_vals = sfh_func(t_assembly)
    dt = t_assembly[1] - t_assembly[0]
    mass_cum = np.cumsum(sfr_vals) * dt
    mass_cum /= mass_cum[-1]
    ax.plot(t_assembly, mass_cum, ls=ls, lw=2, label=label)

# Mark key redshifts
for z_mark in [2.0, 1.0]:
    t_mark = cosmo.age(z_mark)
    ax.axvline(t_mark, ls=':', color='gray', lw=0.8)
    ax.text(t_mark + 0.2, 0.05, r'$z=%g$' % z_mark, color='gray', fontsize=9)

ax.axhline(0.5, ls='--', color='gray', lw=0.5, alpha=0.5)
ax.legend(fontsize=8, loc='lower right')
ax.set_title('Stellar Mass Assembly Histories')
plt.tight_layout()
plt.show()

## Stellar Mass-to-Light RatiosThe key challenge in galaxy observations is converting **luminosity** (what we observe) to **stellar mass** (what we want to know). The conversion factor is the mass-to-light ratio $M_*/L$.For a simple stellar population (SSP):- Young populations are **luminous** and have low $M_*/L \sim 0.1$- Old populations are **faint** per unit mass and have high $M_*/L \sim 3$–$5$The $M_*/L$ ratio correlates tightly with **galaxy color** because both are driven by the age of the stellar population:- Blue galaxies (star-forming) → low $M_*/L$- Red galaxies (quenched) → high $M_*/L$Zibetti et al. (2009) found a useful empirical relation:$$\log_{10}(M_*/L_r) = -0.840 + 1.654 \times (g - r)$$This allows us to estimate $M_*$ from just two observables: $r$-band luminosity and $g-r$ color.

## Exercise 2: Estimating the Milky Way's Stellar MassUse the Zibetti et al. (2009) $M_*/L$–color relation to estimate the stellar mass of the Milky Way.Given (Licquia & Newman 2016):- $M_r = -20.97 + 5\log_{10}(h)$ (absolute $r$-band magnitude)- $g - r = 0.678$- $M_{r,\odot} = 4.68$ (solar absolute magnitude in $r$-band)Steps:1. Compute $M_*/L_r$ from the color2. Compute $L_r$ from the absolute magnitude: $L_r = 10^{0.4(M_{r,\odot} - M_r)}$3. Compute $M_* = (M_*/L_r) \times L_r$4. Compare to the literature value: $M_* = 5.7 \times 10^{10}\,M_\odot$

In [ ]:
# Exercise 2: MW stellar mass from M/L ratioMr_sun = rt.solar_mag['r']  # 4.68Mr_MW = rt.Mr_MWgr_MW = rt.gr_MWprint(f'MW absolute r-band magnitude: M_r = {Mr_MW:.2f}')print(f'MW g-r color: {gr_MW:.3f}')# FILL IN: compute M*/L_r from the Zibetti relationml_r = ...  # FILL IN: rt.mlZibetti(gr_MW)# FILL IN: compute r-band luminosity in solar unitsLr_MW = ...  # FILL IN: 10**(0.4 * (Mr_sun - Mr_MW))# FILL IN: compute stellar massMstar_est = ...  # FILL IN: ml_r * Lr_MWprint(f'\nM*/L_r = {ml_r:.2f} Msun/Lsun')print(f'L_r = {Lr_MW:.2e} Lsun')print(f'M* (estimated) = {Mstar_est:.2e} Msun')print(f'M* (literature) = {rt.Mstar_MW:.2e} Msun')print(f'Ratio: {Mstar_est / rt.Mstar_MW:.2f}')

## The Star Formation Main SequenceOne of the most important observed correlations in galaxy evolution is the **star formation main sequence (SFMS)**: a tight, nearly linear (in log-log) relation between stellar mass and star formation rate for star-forming galaxies.$$\log_{10}(\text{SFR}) \approx 0.8 \times \log_{10}(M_*) - 6.5$$(approximately, at $z \sim 0$; Speagle et al. 2014)Key features:- **Star-forming galaxies** follow the main sequence- **Quenched galaxies** lie below it (the "red and dead" cloud)- The **specific SFR** (sSFR $= \text{SFR}/M_*$) decreases with mass — more massive galaxies are less efficient at forming stars per unit mass- There is a **bimodality** in sSFR: star-forming vs quenched populations

In [ ]:
# Load SDSS galaxy data (672,813 galaxies from MPA-JHU catalog)sdss = rt.loadSdssSubset()print(f'Loaded {len(sdss)} galaxies')print(f'Fields: {sdss.dtype.names}')# Extract useful quantitiesMstar_cat = 10**sdss['lgm_tot_p50']sfr_cat = 10**sdss['sfr_tot_p50']color_gr = sdss['color_gr']Mr_sdss = sdss['M_model_r']z_sdss = sdss['z']

## Exercise 3: The Star Formation Main Sequence from SDSSUsing the SDSS galaxy catalog:1. Plot the **SFR–$M_*$ relation** (the star formation main sequence) as a 2D histogram2. Overplot the Speagle et al. (2014) relation3. Compute and plot the **specific SFR** ($\text{sSFR} = \text{SFR}/M_*$) vs $M_*$4. Plot the **sSFR distribution** to see the bimodality between star-forming and quenched galaxies

In [ ]:
# Exercise 3: Star formation main sequence# Apply quality cutsmask = (Mstar_cat > 0) & (sfr_cat > 0) & (z_sdss > 0.02)mstar = Mstar_cat[mask]sfr = sfr_cat[mask]Vmax_weights = sdss['1/Vmax'][mask]# Set minimum SFR for displaysfr_min = 10**-2.5sfr_plot = np.copy(sfr)sfr_plot[sfr_plot < sfr_min] = sfr_minx = np.log10(mstar)y = np.log10(sfr_plot)x_lo, x_hi = 8.2, 11.8y_lo, y_hi = -2.6, 1.1# FILL IN: create 2D histogram weighted by 1/Vmaxhist, xedges, yedges = ...  # FILL IN: np.histogram2d(x, y, bins=(40, 40), range=[[x_lo, x_hi], [y_lo, y_hi]], weights=Vmax_weights, density=True)hist = np.log10(hist.T[::-1] + 1.0)vmax = np.max(hist) * 0.9fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))# Left: SFR vs M*ax1 = axes[0]ax1.set_xlabel(r'$\log_{10}\ M_*\ (M_\odot)$')ax1.set_ylabel(r'$\log_{10}\ \mathrm{SFR}\ (M_\odot/\mathrm{yr})$')cmap = plt.get_cmap('Blues')cmap.set_under('#FFFFFF')im = ax1.imshow(hist, extent=[x_lo, x_hi, y_lo, y_hi],                interpolation='nearest', aspect='auto',                cmap=cmap, vmin=1e-3, vmax=vmax)ax1.set_xlim(x_lo, x_hi)ax1.set_ylim(y_lo, y_hi)# Speagle+2014 main sequencet0 = cosmo.age(0.15)  # median redshift of SDSS samplex_sp = np.array([x_lo, x_hi])# FILL IN: Speagle+2014 relation: log(SFR) = (0.84 - 0.026*t) * log(M*) - (6.51 - 0.11*t)y_sp = ...  # FILL IN: (0.84 - 0.026 * t0) * x_sp - (6.51 - 0.11 * t0)ax1.plot(x_sp, y_sp, '--', color='orange', lw=2, label='Speagle+2014')ax1.legend(fontsize=10)ax1.set_title('Star Formation Main Sequence')plt.colorbar(im, ax=ax1, label=r'$\log_{10}(dN/N/dx/dy + 1)$')# Right: sSFR vs M*ax2 = axes[1]sfr_raw = np.copy(sfr)sfr_raw[sfr_raw < 1e-10] = 1e-10# FILL IN: compute specific SFRssfr = ...  # FILL IN: np.log10(sfr_raw / mstar)y2_lo, y2_hi = -13.0, -8.8hist2, _, _ = np.histogram2d(x, ssfr, bins=(40, 40),                              range=[[x_lo, x_hi], [y2_lo, y2_hi]],                              weights=Vmax_weights, density=True)hist2 = np.log10(hist2.T[::-1] + 1.0)ax2.set_xlabel(r'$\log_{10}\ M_*\ (M_\odot)$')ax2.set_ylabel(r'$\log_{10}\ \mathrm{sSFR}\ (1/\mathrm{yr})$')im2 = ax2.imshow(hist2, extent=[x_lo, x_hi, y2_lo, y2_hi],                 interpolation='nearest', aspect='auto',                 cmap=cmap, vmin=1e-3, vmax=np.max(hist2) * 0.9)ax2.set_xlim(x_lo, x_hi)ax2.set_ylim(y2_lo, y2_hi)ax2.set_title('Specific SFR')plt.colorbar(im2, ax=ax2, label=r'$\log_{10}(dN/N/dx/dy + 1)$')plt.tight_layout()plt.show()

In [ ]:
# sSFR distribution showing bimodalityplt.figure(figsize=(5, 4))plt.xlabel(r'$\log_{10}\ \mathrm{sSFR}\ (1/\mathrm{yr})$')plt.ylabel(r'$dN/N/dx$')plt.hist(ssfr, range=[-13.0, -8.8], bins=30,         weights=Vmax_weights, density=True,         alpha=0.7, label=r'$V_{\rm max}$-corrected')plt.hist(ssfr, range=[-13.0, -8.8], bins=30,         density=True, histtype='step', color='black',         label='Uncorrected')plt.axvline(-11.0, ls=':', color='gray', lw=0.8,            label='Star-forming / quenched boundary')plt.legend(fontsize=9)plt.title('sSFR Bimodality')plt.tight_layout()plt.show()

## Demonstration: Color–sSFR Correlation

The bimodality in sSFR maps directly onto the **color bimodality** of galaxies. The figure below shows $g-r$ color vs. specific SFR for SDSS galaxies, revealing a remarkably tight correlation: blue galaxies are star-forming, red galaxies are quenched, and the boundary falls near $\text{sSFR} \sim 10^{-11}\,\text{yr}^{-1}$. This is the observational basis for using color as a proxy for stellar population age (adapted from CFN Chapter 7).

In [ ]:
# Color-sSFR correlation from SDSS
# Re-derive quantities in case Exercise 3 hasn't been completed
mask_demo = (Mstar_cat > 0) & (sfr_cat > 0) & (z_sdss > 0.02)
mstar_demo = Mstar_cat[mask_demo]
sfr_demo = sfr_cat[mask_demo]
gr_demo = color_gr[mask_demo]
Vmax_demo = sdss['1/Vmax'][mask_demo]
ssfr_demo = np.log10(sfr_demo / mstar_demo)

fig, ax = plt.subplots(figsize=(5, 4))

x_c, y_c = gr_demo, ssfr_demo
x_lo_c, x_hi_c = 0.0, 1.2
y_lo_c, y_hi_c = -13.0, -8.8

hist_c, _, _ = np.histogram2d(x_c, y_c, bins=(40, 40),
                               range=[[x_lo_c, x_hi_c], [y_lo_c, y_hi_c]],
                               weights=Vmax_demo, density=True)
hist_c = np.log10(hist_c.T[::-1] + 1.0)

cmap_c = plt.get_cmap('Blues')
cmap_c.set_under('#FFFFFF')
im_c = ax.imshow(hist_c, extent=[x_lo_c, x_hi_c, y_lo_c, y_hi_c],
                 interpolation='nearest', aspect='auto',
                 cmap=cmap_c, vmin=1e-3, vmax=np.max(hist_c) * 0.9)

ax.set_xlabel(r'$g - r$')
ax.set_ylabel(r'$\log_{10}\ \mathrm{sSFR}\ (1/\mathrm{yr})$')
ax.axhline(-11.0, ls=':', color='gray', lw=0.8)
ax.text(0.05, -10.7, 'Star-forming', fontsize=9, color='gray')
ax.text(0.05, -11.4, 'Quenched', fontsize=9, color='gray')

plt.colorbar(im_c, ax=ax, label=r'$\log_{10}(dN/N/dx/dy + 1)$')
ax.set_title(r'Color–sSFR Correlation (SDSS)')
plt.tight_layout()
plt.show()

## Summary1. **The IMF** sets the distribution of stellar masses at birth. Chabrier/Kroupa IMFs predict fewer low-mass stars than Salpeter, leading to lower $M_*/L$ ratios.2. **Star formation histories** range from short bursts (small $\tau$) to extended star formation (large $\tau$). The SFH determines how galaxy colors and luminosities evolve.3. **Mass-to-light ratios** depend strongly on stellar age and IMF. The $M_*/L$–color relation allows mass estimation from photometry alone. The Zibetti relation gives $M_* \approx 7.6 \times 10^{10}\,M_\odot$ for the MW.4. **The star formation main sequence** shows that SFR scales with $M_*$ for star-forming galaxies, but more massive galaxies have lower specific SFR. There is a clear bimodality between star-forming and quenched populations.